# Round 3 Manual Bid Optimization

**Product:** Ornamental Bio-Pods (auto-sell at 920 next round)

**Mechanism:**
- **Bid 1**: Immediate — trades with counterparties whose reserve < bid1, you pay bid1
- **Bid 2**: Contingent — trades with counterparties whose reserve ∈ [bid1, bid2), ONLY IF bid2 ≥ avg of all players' bid2. If bid2 < avg_b2 but bid2 > reserve, trade still occurs but PnL is penalized by factor `((920 - avg_b2) / (920 - bid2))^3`

**Counterparty reserves**: Uniform over {670, 675, ..., 920} — 51 counterparties

**Goal**: Find optimal (bid1, bid2) via Monte Carlo simulation over competitor bid2 distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

# ── Parameters ──────────────────────────────────────────────────────────────
SELL_PRICE = 920
RESERVE_MIN, RESERVE_MAX, RESERVE_STEP = 670, 920, 5
N_COMPETITORS = 10   # other players submitting bid2
N_SIMULATIONS = 100_000
RNG = np.random.default_rng(42)

counterparty_reserves = np.arange(RESERVE_MIN, RESERVE_MAX + 1, RESERVE_STEP)  # shape (51,)
print(f"{len(counterparty_reserves)} counterparties: {counterparty_reserves[0]} → {counterparty_reserves[-1]}")

## Profit Functions

In [ ]:
def bid1_profit(bid1: float) -> float:
    """Deterministic profit from bid1 trades."""
    eligible = counterparty_reserves[counterparty_reserves < bid1]
    return (SELL_PRICE - bid1) * len(eligible)


def bid2_profit_single(bid1: float, bid2: float, avg_b2: float) -> float:
    """Profit from bid2 trades given a realized avg_b2."""
    eligible = counterparty_reserves[
        (counterparty_reserves >= bid1) & (counterparty_reserves < bid2)
    ]
    n = len(eligible)
    if n == 0 or bid2 >= SELL_PRICE:
        return 0.0
    margin = SELL_PRICE - bid2
    if bid2 >= avg_b2:
        return margin * n
    else:
        penalty = ((SELL_PRICE - avg_b2) / (SELL_PRICE - bid2)) ** 3
        return margin * n * penalty


# Sanity checks
assert bid1_profit(670) == 0.0,   "bid1=670 buys nobody"
assert bid1_profit(920) == 0.0,   "bid1=920 → zero margin"
assert bid2_profit_single(670, 750, 730) > 0
print("Sanity checks passed")
print(f"bid1=800 profit: {bid1_profit(800):.1f}")

## Bid 1 Analysis (Deterministic)

In [ ]:
bid1_grid = np.arange(670, 921, 5)
bid1_profits = [bid1_profit(b) for b in bid1_grid]

optimal_b1 = bid1_grid[np.argmax(bid1_profits)]
print(f"Optimal bid1 (standalone): {optimal_b1}  →  profit = {max(bid1_profits):.1f}")

plt.figure(figsize=(9, 4))
plt.plot(bid1_grid, bid1_profits, lw=2, color='steelblue')
plt.axvline(optimal_b1, color='red', ls='--', label=f'Optimal bid1 = {optimal_b1}')
plt.xlabel('Bid 1')
plt.ylabel('Profit')
plt.title('Bid 1 Profit (Deterministic)')
plt.legend()
plt.tight_layout()
plt.show()

## Monte Carlo: Competitor Bid2 Distribution

We don't know how competitors choose bid2. We model three scenarios:
- **Uniform**: competitors draw uniformly over the valid range
- **Aggressive**: competitors cluster near high values (Normal centered near 900)
- **Conservative**: competitors cluster near low values (Normal centered near 750)

In [ ]:
def sample_avg_b2(scenario: str, n_sims: int) -> np.ndarray:
    """Sample N_SIMULATIONS realizations of avg_b2 under different competitor assumptions."""
    if scenario == 'uniform':
        draws = RNG.uniform(670, 920, size=(n_sims, N_COMPETITORS))
    elif scenario == 'aggressive':
        draws = RNG.normal(loc=870, scale=30, size=(n_sims, N_COMPETITORS)).clip(670, 919)
    elif scenario == 'conservative':
        draws = RNG.normal(loc=750, scale=40, size=(n_sims, N_COMPETITORS)).clip(670, 919)
    else:
        raise ValueError(scenario)
    # avg_b2 includes our own bid2 — we add it when we evaluate each (bid1, bid2) pair
    return draws  # shape (n_sims, N_COMPETITORS)

competitor_draws = {s: sample_avg_b2(s, N_SIMULATIONS) for s in ['uniform', 'aggressive', 'conservative']}
print("Competitor draw samples ready")

In [ ]:
def expected_profit(bid1: float, bid2: float, competitor_b2_draws: np.ndarray) -> dict:
    """
    Compute stats for a (bid1, bid2) pair.
    competitor_b2_draws: shape (n_sims, N_COMPETITORS) — other players' bid2s
    """
    # avg_b2 = mean of (competitors + our own bid2)
    our_col = np.full((N_SIMULATIONS, 1), bid2)
    all_b2 = np.concatenate([competitor_b2_draws, our_col], axis=1)  # (n_sims, N_COMPETITORS+1)
    avg_b2 = all_b2.mean(axis=1)  # (n_sims,)

    b1_p = bid1_profit(bid1)  # scalar, same every sim

    eligible = counterparty_reserves[
        (counterparty_reserves >= bid1) & (counterparty_reserves < bid2)
    ]
    n_eligible = len(eligible)
    margin = SELL_PRICE - bid2

    no_penalty = np.ones(N_SIMULATIONS, dtype=bool)  # default: no penalty applies
    if n_eligible == 0 or margin <= 0:
        b2_profits = np.zeros(N_SIMULATIONS)
    else:
        no_penalty = bid2 >= avg_b2  # boolean array
        penalty = ((SELL_PRICE - avg_b2) / (SELL_PRICE - bid2)) ** 3
        b2_profits = np.where(no_penalty, margin * n_eligible, margin * n_eligible * penalty)

    total = b1_p + b2_profits
    return {
        'mean': float(total.mean()),
        'std':  float(total.std()),
        'p5':   float(np.percentile(total, 5)),
        'p_no_penalty': float(no_penalty.mean()) if n_eligible > 0 else 1.0,
    }

# Quick test
test = expected_profit(780, 870, competitor_draws['uniform'])
print(f"Test (bid1=780, bid2=870, uniform): E={test['mean']:.1f}, std={test['std']:.1f}, P(no_penalty)={test['p_no_penalty']:.2%}")

## Grid Search

In [ ]:
bid1_values = np.arange(670, 916, 5)   # bid1 candidates
bid2_values = np.arange(670, 921, 5)   # bid2 candidates

results = {}  # scenario → dict of (bid1, bid2) → stats

for scenario, draws in competitor_draws.items():
    print(f"Running grid search: {scenario}...", end=' ', flush=True)
    scenario_results = {}

    # Precompute avg_b2 for each bid2 value once — avoids concatenating a
    # 100K×11 array inside the inner loop (was ~3825 allocations per scenario).
    competitor_sum = draws.sum(axis=1)  # (N_SIMULATIONS,)
    avg_b2_cache = {
        int(b2): (competitor_sum + b2) / (N_COMPETITORS + 1)
        for b2 in bid2_values
    }

    for b1 in bid1_values:
        b1_p = bid1_profit(b1)
        for b2 in bid2_values:
            if b2 <= b1:
                continue
            avg_b2 = avg_b2_cache[int(b2)]
            eligible = counterparty_reserves[
                (counterparty_reserves >= b1) & (counterparty_reserves < b2)
            ]
            n_eligible = len(eligible)
            margin = float(SELL_PRICE - b2)

            if n_eligible == 0 or margin <= 0:
                b2_profits = np.zeros(N_SIMULATIONS)
                p_no_pen = 1.0
            else:
                no_penalty = b2 >= avg_b2
                penalty = ((SELL_PRICE - avg_b2) / margin) ** 3
                b2_profits = np.where(no_penalty, margin * n_eligible,
                                      margin * n_eligible * penalty)
                p_no_pen = float(no_penalty.mean())

            total = b1_p + b2_profits
            scenario_results[(int(b1), int(b2))] = {
                'mean': float(total.mean()),
                'std':  float(total.std()),
                'p5':   float(np.percentile(total, 5)),
                'p_no_penalty': p_no_pen,
            }

    results[scenario] = scenario_results
    best = max(scenario_results, key=lambda k: scenario_results[k]['mean'])
    print(f"done. Best: bid1={best[0]}, bid2={best[1]}, E[profit]={scenario_results[best]['mean']:.1f}")

## Heatmaps: E[Profit] by (bid1, bid2)

In [ ]:
def make_heatmap_matrix(scenario_results, stat='mean'):
    b1s = sorted(set(k[0] for k in scenario_results))
    b2s = sorted(set(k[1] for k in scenario_results))
    mat = np.full((len(b1s), len(b2s)), np.nan)
    b1_idx = {v: i for i, v in enumerate(b1s)}
    b2_idx = {v: i for i, v in enumerate(b2s)}
    for (b1, b2), stats in scenario_results.items():
        mat[b1_idx[b1], b2_idx[b2]] = stats[stat]
    return mat, b1s, b2s


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, scenario in zip(axes, ['uniform', 'aggressive', 'conservative']):
    mat, b1s, b2s = make_heatmap_matrix(results[scenario], 'mean')
    im = ax.imshow(mat, aspect='auto', origin='lower',
                   extent=[b2s[0], b2s[-1], b1s[0], b1s[-1]],
                   cmap='RdYlGn')
    # Mark optimal
    best = max(results[scenario], key=lambda k: results[scenario][k]['mean'])
    ax.scatter(best[1], best[0], marker='*', s=200, c='white', zorder=5, label=f'Best ({best[0]},{best[1]})')
    ax.set_xlabel('Bid 2')
    ax.set_ylabel('Bid 1')
    ax.set_title(f'E[Profit] — {scenario} competitors')
    ax.legend(fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Bid 2 Sweep: Fixed Bid 1, Vary Bid 2

In [ ]:
# Pick the best bid1 under uniform scenario, then sweep bid2
best_uniform = max(results['uniform'], key=lambda k: results['uniform'][k]['mean'])
fixed_b1 = best_uniform[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scenario in zip(axes, ['uniform', 'aggressive']):
    b2_sweep = [b2 for b2 in bid2_values if b2 > fixed_b1]
    means = [results[scenario].get((fixed_b1, b2), {}).get('mean', np.nan) for b2 in b2_sweep]
    stds  = [results[scenario].get((fixed_b1, b2), {}).get('std',  np.nan) for b2 in b2_sweep]
    means, stds = np.array(means), np.array(stds)

    ax.plot(b2_sweep, means, lw=2, label='E[profit]')
    ax.fill_between(b2_sweep, means - stds, means + stds, alpha=0.25, label='±1 std')
    best_b2 = b2_sweep[np.nanargmax(means)]
    ax.axvline(best_b2, color='red', ls='--', label=f'Best bid2 = {best_b2}')
    ax.set_xlabel('Bid 2')
    ax.set_ylabel('Total Profit')
    ax.set_title(f'bid1={fixed_b1}, {scenario} competitors')
    ax.legend()

plt.tight_layout()
plt.show()

## Risk–Return: E[Profit] vs Std

In [ ]:
scenario = 'uniform'
means = np.array([v['mean'] for v in results[scenario].values()])
stds  = np.array([v['std']  for v in results[scenario].values()])
keys  = list(results[scenario].keys())

plt.figure(figsize=(9, 5))
sc = plt.scatter(stds, means, c=means, cmap='RdYlGn', alpha=0.5, s=20)
plt.colorbar(sc, label='E[Profit]')

# Highlight Pareto frontier (max mean for each std bucket)
best = max(keys, key=lambda k: results[scenario][k]['mean'])
bv = results[scenario][best]
plt.scatter(bv['std'], bv['mean'], s=200, marker='*', c='black', zorder=5,
            label=f'Global best: bid1={best[0]}, bid2={best[1]}')

plt.xlabel('Std of Profit')
plt.ylabel('E[Profit]')
plt.title('Risk–Return for all (bid1, bid2) pairs — uniform competitors')
plt.legend()
plt.tight_layout()
plt.show()

## Top Recommendations

In [ ]:
for scenario in ['uniform', 'aggressive', 'conservative']:
    print(f"\n=== {scenario.upper()} COMPETITORS ===")
    top5 = sorted(results[scenario].items(), key=lambda x: x[1]['mean'], reverse=True)[:5]
    print(f"{'bid1':>6} {'bid2':>6} {'E[profit]':>10} {'std':>8} {'p5':>8} {'P(no pen)':>10}")
    for (b1, b2), s in top5:
        print(f"{b1:>6} {b2:>6} {s['mean']:>10.1f} {s['std']:>8.1f} {s['p5']:>8.1f} {s['p_no_penalty']:>10.2%}")

## Sensitivity: How Does Optimal Bid2 Shift with Competitor Aggressiveness?

In [ ]:
competitor_means = np.arange(700, 920, 10)  # sweep competitor avg bid2 center
optimal_b2_by_center = []

for center in competitor_means:
    draws = RNG.normal(loc=center, scale=40, size=(N_SIMULATIONS, N_COMPETITORS)).clip(670, 919)
    best_profit = -np.inf
    best_b2 = None
    fixed_b1 = fixed_b1  # use same bid1 from above
    for b2 in bid2_values:
        if b2 <= fixed_b1:
            continue
        stats = expected_profit(fixed_b1, b2, draws)
        if stats['mean'] > best_profit:
            best_profit = stats['mean']
            best_b2 = b2
    optimal_b2_by_center.append(best_b2)

plt.figure(figsize=(9, 4))
plt.plot(competitor_means, optimal_b2_by_center, marker='o', lw=2)
plt.xlabel('Competitor Bid2 Distribution Mean')
plt.ylabel(f'Optimal Bid2 (bid1={fixed_b1} fixed)')
plt.title('Optimal Bid2 vs Competitor Aggressiveness')
plt.tight_layout()
plt.show()